# Search request with query + selected tourism tag

In [14]:
import requests
import pprint
from bs4 import BeautifulSoup

page_offset = 20
page = 1

url = "https://ria.ru/services/search/getmore/"
params = {
    "query": "Россия",
    "offset": (page - 1) * page_offset,
    "list_sids[]": "tourism",
    "sort[]": "date"
}

response = requests.get(url, params=params)

assert response.status_code == 200

soup = BeautifulSoup(response.text, "html.parser")

list_items_loaded = soup.find("div", class_="list-items-loaded")
total = int(list_items_loaded["data-count"]) if list_items_loaded and list_items_loaded.has_attr("data-count") else None
page_size = page_offset
current_page = page

items = []
for item_div in soup.find_all("div", class_="list-item"):
    title_a = item_div.find("a", class_="list-item__title")
    title = title_a.get_text(strip=True) if title_a else None
    link = title_a["href"] if title_a and title_a.has_attr("href") else None

    time_div = item_div.find("div", class_="list-item__info-item", attrs={"data-type": "date"})
    time = time_div.get_text(strip=True) if time_div else None

    views_div = item_div.find("div", class_="list-item__info-item", attrs={"data-type": "views"})
    views = None
    if views_div:
        views_span = views_div.find("span")
        views = views_span.get_text(strip=True) if views_span else None

    assets = []
    img_link = item_div.find("a", class_="list-item__image")
    if img_link:
        imgs = img_link.find_all("img")
        assets = [img["src"] for img in imgs if img.has_attr("src")]

    items.append({
        "title": title,
        "time": time,
        "views": views,
        "assets": assets,
        "url": link
    })

pagination_model = {
    "page": current_page,
    "page_size": page_size,
    "total": total,
    "items": items
}

pprint.pprint(pagination_model)

{'items': [{'assets': ['https://cdnn21.img.ria.ru/images/07ea/03/0c/2080322209_0:161:3068:1887_650x0_80_0_0_092c0e81a56ef187d03a2ba7dd2b7626.jpg'],
            'time': '18 марта, 20:35',
            'title': 'Власти Антальи не получали жалоб от россиян на '
                     'безопасность отдыха',
            'url': 'https://ria.ru/20260318/turtsiya-2081544892.html',
            'views': '4415'},
           {'assets': ['https://cdnn21.img.ria.ru/images/07e9/0c/04/2059810121_0:512:2730:2048_650x0_80_0_0_f05c12dcb7a7c0fb8190d2adaccf1067.jpg'],
            'time': '18 марта, 19:49',
            'title': 'Восемь регионов "Золотого кольца" подписали соглашение о '
                     'развитии маршрута',
            'url': 'https://ria.ru/20260318/koltso-2081537838.html',
            'views': '1946'},
           {'assets': ['https://cdnn21.img.ria.ru/images/155899/61/1558996144_0:221:2856:1828_650x0_80_0_0_288b1d934ba6dfa7042140a047587352.jpg'],
            'time': '18 марта, 19:48',
  